# 4. 평가 및 시각화

모델 성능을 다양한 지표로 평가하고, 시각화를 통해 모델의 강점과 약점을 분석합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', family='Apple SD Gothic Neo')
plt.rc('axes', unicode_minus=False)

train_data = pd.read_csv('../data/train_processed.csv')
X = train_data.drop('label', axis=1)
y = train_data['label']

## 4.1 모델별 성능 비교 (Accuracy, Precision, Recall, F1-score)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'LightGBM': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=42, use_label_encoder=False, eval_metric='logloss'),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
eval_results = []

for name, model in models.items():
    fold_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = model.predict(X.iloc[val_idx])
        fold_metrics['accuracy'].append(accuracy_score(y.iloc[val_idx], pred))
        fold_metrics['precision'].append(precision_score(y.iloc[val_idx], pred))
        fold_metrics['recall'].append(recall_score(y.iloc[val_idx], pred))
        fold_metrics['f1'].append(f1_score(y.iloc[val_idx], pred))

    eval_results.append({
        'Model': name,
        'Accuracy': np.mean(fold_metrics['accuracy']),
        'Precision': np.mean(fold_metrics['precision']),
        'Recall': np.mean(fold_metrics['recall']),
        'F1-score': np.mean(fold_metrics['f1']),
    })

eval_df = pd.DataFrame(eval_results).set_index('Model')
eval_df

## 4.2 성능 비교 시각화

In [ ]:
eval_df.plot(kind='bar', figsize=(10, 5), rot=0)
plt.title('모델별 성능 비교')
plt.ylabel('Score')
plt.ylim(0.5, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4.3 Confusion Matrix (최적 모델)

In [ ]:
# 가장 성능이 좋은 모델로 Confusion Matrix 시각화
best_model_name = eval_df['F1-score'].idxmax()
best_model = models[best_model_name]
best_model.fit(X, y)

# 마지막 fold의 validation으로 시각화
for tr_idx, val_idx in skf.split(X, y):
    pass  # 마지막 fold 사용
best_model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
val_pred = best_model.predict(X.iloc[val_idx])

cm = confusion_matrix(y.iloc[val_idx], val_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Smoker', 'Smoker'],
            yticklabels=['Non-Smoker', 'Smoker'])
plt.title(f'Confusion Matrix ({best_model_name})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(classification_report(y.iloc[val_idx], val_pred, target_names=['Non-Smoker', 'Smoker']))

## 4.4 ROC Curve

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in models.items():
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X.iloc[val_idx])[:, 1]
    fpr, tpr, _ = roc_curve(y.iloc[val_idx], proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 4.5 Feature Importance (Top 20)

In [ ]:
best_model.fit(X, y)
importance = best_model.feature_importances_
feat_imp = pd.Series(importance, index=X.columns).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
feat_imp.sort_values().plot(kind='barh', color='steelblue')
plt.title(f'Feature Importance Top 20 ({best_model_name})')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()